<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/PDF_Signer_And_Hasher_Auth_Audit_EmailFixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ PDF_Signer_And_Hasher (Auth + Audit + Email Extraction Fixed)
**Kapodistrian Internal Signing Tools** – Secure + Smart Version

Includes:
- 🔐 Local signer authentication
- 🧠 Auto extraction of email with override
- ✍️ Visible footer with full SHA256
- 🧾 Audit log of each signing

In [1]:
# 🔐 Signer Authentication
from getpass import getpass
from datetime import datetime

valid_signers = {
    "aval": "kas123",
    "jkar": "sig2025",
    "kgeo": "audit99"
}

signer_id = input("Enter signer ID (e.g. initials): ").strip().lower()
signer_pwd = getpass("Enter signer passcode: ").strip()

if signer_id not in valid_signers or valid_signers[signer_id] != signer_pwd:
    raise PermissionError("❌ Unauthorized signer. Audit blocked.")
else:
    print("✅ Signer authenticated:", signer_id)

Enter signer ID (e.g. initials): aval
Enter signer passcode: ··········
✅ Signer authenticated: aval


In [2]:
# 🔧 Install PyMuPDF
!pip install --quiet PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 54.7 MB/s eta 0:00:00


In [3]:
# 📤 Upload PDF
from google.colab import files
uploaded = files.upload()
pdf_filename = [f for f in uploaded if f.endswith('.pdf')][0]

Saving Spectral_Realization_of_the_Nontrivial_Zeros_of_the_Riemann_Zeta_Function_via_a_Hermitian_Operator_Framework.pdf to Spectral_Realization_of_the_Nontrivial_Zeros_of_the_Riemann_Zeta_Function_via_a_Hermitian_Operator_Framework.pdf


In [4]:
# 🧠 Metadata collection with email extraction
import fitz
import re

doc = fitz.open(pdf_filename)
full_text = "\n".join([page.get_text() for page in doc])

email_match = re.search(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", full_text)
auto_email = email_match.group(0).strip() if email_match else ""

author = input("Author name: ").strip()
title = input("Short title: ").strip()
version = input("Version (e.g. v1): ").strip()
orcid = input("ORCID: ").strip()
institution = input("Institution: ").strip()
email = input(f"Email [{auto_email}]: ").strip() or auto_email

date_str = datetime.today().strftime('%Y-%m-%d')
kas_filename = f"{author}_{title}_{version}_SIGNED_KAS_{date_str}.pdf"

Author name: Antonios Valamontes
Short title: 
Version (e.g. v1): v1.0
ORCID: 0009-0008-5616-7746
Institution: Kapodistrian Academy of Science
Email [avalamontes@kapodistrian.edu.gr]: 


In [5]:
# ✍️ Add footer and hash
footer = f"Digitally Signed by: {author}\nORCID: {orcid}\nInstitution: {institution}\nEmail: {email}\nDate: {date_str}\nSHA256: {{hash_placeholder}}\nVerified by the Kapodistrian Academy of Science"

# Save original PDF temporarily to compute hash
temp_path = "_temp_original.pdf"
doc.save(temp_path)

import hashlib
with open(temp_path, 'rb') as f:
    sha256_hash = hashlib.sha256(f.read()).hexdigest()

# Embed footer
footer = footer.replace("{hash_placeholder}", sha256_hash)
page = doc[-1]
rect = fitz.Rect(36, page.rect.height - 110, page.rect.width - 36, page.rect.height - 20)
page.insert_textbox(rect, footer, fontsize=8, color=(0, 0, 0))
doc.save(kas_filename)

In [6]:
# 💾 Save .sha256 and .meta.txt
sha_filename = kas_filename + ".sha256"
with open(sha_filename, 'w') as f:
    f.write(sha256_hash)

meta_filename = kas_filename.replace(".pdf", ".meta.txt")
with open(meta_filename, "w") as f:
    f.write(f"Author: {author}\n")
    f.write(f"Title: {title}\n")
    f.write(f"Version: {version}\n")
    f.write(f"Date: {date_str}\n")
    f.write(f"ORCID: {orcid}\n")
    f.write(f"Institution: {institution}\n")
    f.write(f"Email: {email}\n")
    f.write(f"SHA256: {sha256_hash}\n")

In [7]:
# 🧾 Write to audit log
audit_entry = f"[{datetime.utcnow().isoformat()} UTC] Signer: {signer_id}\n"
audit_entry += f"  File: {kas_filename}\n  SHA256: {sha256_hash}\n  Author: {author}, ORCID: {orcid}, Email: {email}, Institution: {institution}\n"
audit_entry += "  Status: ✅ Signed successfully\n\n"

with open("KAS_audit.log", "a") as f:
    f.write(audit_entry)

print("🔐 Audit entry recorded.")

🔐 Audit entry recorded.


In [8]:
# 📁 Download results
files.download(kas_filename)
files.download(sha_filename)
files.download(meta_filename)
files.download("KAS_audit.log")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>